In [ ]:
import pandas as pd, numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score

# --- Load data (your paths) ---
pos = pd.read_csv("/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/positive_values/merged_TSS_500bp_20bins.csv")
neg = pd.read_csv("/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negative_values/ATAC_negatives_2k_features_TSS_only.csv")

pos["label"] = 1
neg["label"] = 0
data = pd.concat([pos, neg], ignore_index=True)

# Keep numeric features only
X_df = data.drop(columns=["label"])
X_df = X_df.select_dtypes(include=["number"])
y = data["label"].astype(int)
feature_names = X_df.columns.tolist()

print("Total features:", X_df.shape[1])

# --- Optional transform; tree models don't need scaling, but we’ll keep your setup ---
X = np.log1p(X_df.clip(lower=0))  # clip in case of tiny negatives

# Train/test split (hold-out test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Standardization (not required for trees, harmless though)
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Inner split for model selection of K (avoid peeking at test set)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, stratify=y_train, random_state=42
)

# Base XGB config (used both for selector and final model)
xgb_base_params = dict(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    tree_method="hist",
    importance_type="gain",
    n_jobs=-1,
    random_state=42,
    eval_metric="logloss"
)

# --- Search best top-K using embedded selector on the inner validation split ---
K_LIST = [50, 100, 200, 300, None]  # None = keep all (after any cleaning)
best_k, best_ap = None, -np.inf
best_selector = None

for k in K_LIST:
    selector = SelectFromModel(
        estimator=XGBClassifier(**xgb_base_params),
        threshold=-np.inf,            # disable thresholding by value
        max_features=k                # cap by top-K importances
    )
    selector.fit(X_tr, y_tr)         # trains internal XGB to get importances
    X_tr_sel  = selector.transform(X_tr)
    X_val_sel = selector.transform(X_val)

    # If nothing selected, skip
    if X_tr_sel.shape[1] == 0:
        continue

    clf = XGBClassifier(**xgb_base_params)
    clf.fit(X_tr_sel, y_tr)
    val_proba = clf.predict_proba(X_val_sel)[:, 1]
    ap = average_precision_score(y_val, val_proba)

    print(f"K={k}: selected={X_tr_sel.shape[1]}  Val PR AUC={ap:.4f}")
    if ap > best_ap:
        best_ap, best_k, best_selector = ap, k, selector

if best_selector is None:
    raise RuntimeError("No features were selected. Try larger K or check data preprocessing.")

print(f"\nBest K={best_k} with Val PR AUC={best_ap:.4f}")

# --- Fit selector on the full training set with the chosen K ---
selector_final = SelectFromModel(
    estimator=XGBClassifier(**xgb_base_params),
    threshold=-np.inf,
    max_features=best_k
)
selector_final.fit(X_train_scaled, y_train)

# Recover selected feature names
support = selector_final.get_support()
selected_features = [name for name, keep in zip(feature_names, support) if keep]
print(f"Selected {len(selected_features)} features.")
# Save list if you want:
pd.Series(selected_features).to_csv("selected_features_xgb_topk.txt", index=False)

# Transform train/test
X_train_sel = selector_final.transform(X_train_scaled)
X_test_sel  = selector_final.transform(X_test_scaled)

# --- Train final classifier on selected features ---
xgb_final = XGBClassifier(**xgb_base_params | {"n_estimators": 600})
xgb_final.fit(X_train_sel, y_train)

# --- Evaluate on hold-out test ---
y_proba = xgb_final.predict_proba(X_test_sel)[:, 1]
y_pred  = (y_proba >= 0.5).astype(int)

print("\nClassification report (threshold=0.5):")
print(classification_report(y_test, y_pred, digits=4))

print("Test PR AUC:", average_precision_score(y_test, y_proba))
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

# Probabilities from your final model
y_score = xgb_final.predict_proba(X_test_sel)[:, 1]

# ROC
fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc = roc_auc_score(y_test, y_score)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.3f}")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# PR
prec, rec, thr = precision_recall_curve(y_test, y_score)
ap = average_precision_score(y_test, y_score)
plt.figure(figsize=(6,4))
plt.plot(rec, prec, label=f"AP = {ap:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curve")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import f1_score

# Evaluate F1 across thresholds
thresholds = np.linspace(0.0, 1.0, 201)
f1s = []
for t in thresholds:
    y_pred_t = (y_score >= t).astype(int)
    f1s.append(f1_score(y_test, y_pred_t) if (y_pred_t.sum() and (1-y_pred_t).sum()) or True else 0)

best_idx = int(np.argmax(f1s))
best_t = float(thresholds[best_idx])
best_f1 = float(f1s[best_idx])

plt.figure(figsize=(6,4))
plt.plot(thresholds, f1s)
plt.axvline(best_t, linestyle="--")
plt.xlabel("Threshold")
plt.ylabel("F1 score")
plt.title(f"F1 vs threshold (best={best_t:.3f}, F1={best_f1:.3f})")
plt.tight_layout()
plt.show()

print(f"Chosen threshold: {best_t:.3f}")


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = (y_score >= best_t).astype(int)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(4,4))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.xticks([0,1], ["Pred 0","Pred 1"])
plt.yticks([0,1], ["True 0","True 1"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, digits=4))


In [ ]:
import pandas as pd
# Map model importances (aligned to selected feature order) back to names
imp = pd.Series(xgb_final.feature_importances_, index=selected_features)
imp_sorted = imp.sort_values(ascending=False)

TOPK = 30 if imp_sorted.shape[0] >= 30 else imp_sorted.shape[0]
top_imp = imp_sorted.head(TOPK)

plt.figure(figsize=(8, max(4, 0.28*TOPK)))
top_imp.sort_values().plot(kind="barh")
plt.xlabel("Gain importance")
plt.title(f"Top {TOPK} features")
plt.tight_layout()
plt.show()

# Save for records
imp_sorted.to_csv("xgb_final_feature_importances.csv", header=["importance"])


In [ ]:
# import TSS data 
import pandas as pd, numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score

# --- Load data (your paths) ---
pos = pd.read_csv("/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/positive_values/merged_TSS_500bp_20bins.csv")
neg = pd.read_csv("/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negative_values/ATAC_negatives_2k_features_TSS_only.csv")

pos["label"] = 1
neg["label"] = 0
data = pd.concat([pos, neg], ignore_index=True)

# Build features (drop label + TSS_coord if present)
X_df = data.drop(columns=[c for c in ["label", "TSS_coord"] if c in data.columns], errors="ignore")
X_df = X_df.select_dtypes(include=["number"])
feature_names = X_df.columns.tolist()

# --- Optional transform; tree models don't need scaling, but we’ll keep your setup ---
X = np.log1p(X_df.clip(lower=0))  # clip in case of tiny negatives

# Train/test split (hold-out test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Standardization (not required for trees, harmless though)
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Inner split for model selection of K (avoid peeking at test set)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, stratify=y_train, random_state=42
)

# Base XGB config (used both for selector and final model)
xgb_base_params = dict(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    tree_method="hist",
    importance_type="gain",
    n_jobs=-1,
    random_state=42,
    eval_metric="logloss"
)

# --- Search best top-K using embedded selector on the inner validation split ---
K_LIST = [50, 100, 200, 300, None]  # None = keep all (after any cleaning)
best_k, best_ap = None, -np.inf
best_selector = None

for k in K_LIST:
    selector = SelectFromModel(
        estimator=XGBClassifier(**xgb_base_params),
        threshold=-np.inf,            # disable thresholding by value
        max_features=k                # cap by top-K importances
    )
    selector.fit(X_tr, y_tr)         # trains internal XGB to get importances
    X_tr_sel  = selector.transform(X_tr)
    X_val_sel = selector.transform(X_val)

    # If nothing selected, skip
    if X_tr_sel.shape[1] == 0:
        continue

    clf = XGBClassifier(**xgb_base_params)
    clf.fit(X_tr_sel, y_tr)
    val_proba = clf.predict_proba(X_val_sel)[:, 1]
    ap = average_precision_score(y_val, val_proba)

    print(f"K={k}: selected={X_tr_sel.shape[1]}  Val PR AUC={ap:.4f}")
    if ap > best_ap:
        best_ap, best_k, best_selector = ap, k, selector

if best_selector is None:
    raise RuntimeError("No features were selected. Try larger K or check data preprocessing.")

print(f"\nBest K={best_k} with Val PR AUC={best_ap:.4f}")

# --- Fit selector on the full training set with the chosen K ---
selector_final = SelectFromModel(
    estimator=XGBClassifier(**xgb_base_params),
    threshold=-np.inf,
    max_features=best_k
)
selector_final.fit(X_train_scaled, y_train)

# Recover selected feature names
support = selector_final.get_support()
selected_features = [name for name, keep in zip(feature_names, support) if keep]
print(f"Selected {len(selected_features)} features.")
# Save list if you want:
pd.Series(selected_features).to_csv("selected_features_xgb_topk.txt", index=False)

# Transform train/test
X_train_sel = selector_final.transform(X_train_scaled)
X_test_sel  = selector_final.transform(X_test_scaled)

# --- Train final classifier on selected features ---
xgb_final = XGBClassifier(**xgb_base_params | {"n_estimators": 600})
xgb_final.fit(X_train_sel, y_train)

# --- Evaluate on hold-out test ---
y_proba = xgb_final.predict_proba(X_test_sel)[:, 1]
y_pred  = (y_proba >= 0.5).astype(int)

print("\nClassification report (threshold=0.5):")
print(classification_report(y_test, y_pred, digits=4))

print("Test PR AUC:", average_precision_score(y_test, y_proba))
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))


In [ ]:
import pandas as pd
# Map model importances (aligned to selected feature order) back to names
imp = pd.Series(xgb_final.feature_importances_, index=selected_features)
imp_sorted = imp.sort_values(ascending=False)

TOPK = 30 if imp_sorted.shape[0] >= 30 else imp_sorted.shape[0]
top_imp = imp_sorted.head(TOPK)

plt.figure(figsize=(8, max(4, 0.28*TOPK)))
top_imp.sort_values().plot(kind="barh")
plt.xlabel("Gain importance")
plt.title(f"Top {TOPK} features")
plt.tight_layout()
plt.show()

# Save for records
imp_sorted.to_csv("xgb_final_feature_importances.csv", header=["importance"])


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.feature_selection import f_classif
from sklearn.impute import SimpleImputer
from statsmodels.stats.multitest import multipletests

# X_df: numeric features DataFrame
# y:    labels (0/1 or multi-class)

def rank_features_anova(X_df, y, save_csv="anova_feature_ranking.csv", topk=30):
    # Ensure numeric matrix & handle missing values
    X_num = X_df.select_dtypes(include=["number"]).copy()
    imp = SimpleImputer(strategy="median")
    X_imp = imp.fit_transform(X_num)

    # ANOVA F-test (one-way, per feature)
    F, p = f_classif(X_imp, y.values)
    # Make robust to NaNs/Infs (can occur for constant columns)
    F = np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)
    p = np.nan_to_num(p, nan=1.0, posinf=1.0, neginf=1.0)

    # FDR (Benjamini–Hochberg) and effect size η²
    q = multipletests(p, method="fdr_bh")[1]
    k = y.nunique()
    N = len(y)
    eta2 = (F * (k - 1)) / (F * (k - 1) + (N - k) + 1e-12)

    anova_df = pd.DataFrame({
        "feature": X_num.columns,
        "F": F,
        "p": p,
        "q_fdr": q,
        "eta2": eta2
    }).sort_values("F", ascending=False).reset_index(drop=True)

    # Save
    if save_csv:
        anova_df.to_csv(save_csv, index=False)
        print(f"Saved ANOVA ranking -> {save_csv}")

    # Quick look & plot
    print(anova_df.head(10))
    top = anova_df.head(min(topk, len(anova_df)))
    plt.figure(figsize=(8, max(4, 0.28*len(top))))
    plt.barh(top["feature"][::-1], top["F"][::-1])
    plt.xlabel("ANOVA F")
    plt.title("Top features by ANOVA F", fontsize=16, pad=6)
    plt.tight_layout()
    plt.show()
    return anova_df

anova_table = rank_features_anova(X_df, y, save_csv="anova_feature_ranking.csv", topk=30)

